# Clustering Comparison: Fashion-MNIST

Comparing all 5 methods on Fashion-MNIST (`data/fashion-mnist_train.csv` +
`data/fashion-mnist_test.csv`, 70,000 28x28 grayscale clothing images, 10
classes as ground truth).

**Scale note:** Spectral Clustering and DBSCAN/HDBSCAN don't scale to 70k
points (O(n^2) or worse in time/memory). We take a **stratified subsample**
(~3,000 points, balanced across the 10 classes) after reducing pixels to 50
PCA components, so all 5 methods run in reasonable time and are directly
comparable. Results describe this subsample, not the full dataset.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_fashion_mnist
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "calinski_harabasz",
]
EXTERNAL_COLS = ["ari", "nmi", "ami", "fmi"]

In [ ]:
ds = load_fashion_mnist(n_samples=3000, n_pca=50, random_state=42)
print(f"{ds.name}: X={ds.data.shape}, classes={ds.target_names}, n_true_clusters={ds.n_true_clusters}")
print(f"PCA explained variance (50 components): {ds.explained_variance_ratio.sum():.3f}")

## Exploratory look

In [ ]:
labels, counts = np.unique(ds.target, return_counts=True)
pd.Series(counts, index=[ds.target_names[i] for i in labels], name="count").sort_values(ascending=False)

In [ ]:
# A few example images per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls, ax in zip(range(10), axes.flat):
    idx = np.where(ds.target == cls)[0][0]
    ax.imshow(ds.pixels[idx].reshape(28, 28), cmap="gray")
    ax.set_title(ds.target_names[cls], fontsize=9)
    ax.axis("off")
fig.suptitle("One example per class")
plt.tight_layout()
plt.show()

## 2D embedding for visualization

Clustering runs on the 50-D PCA space; a further PCA to 2D here is only for
plotting.

In [ ]:
X_2d, xlabel, ylabel = gf.project_2d(ds.data)

## DBSCAN eps selection

In [ ]:
ms = tuning.default_min_samples(ds.data.shape[1])
curve = tuning.k_distance_curve(ds.data, ms)
knee_idx, eps_at_knee = tuning.knee(curve)
gf.plot_k_distance(curve, knee_idx, eps_at_knee, min_samples=ms)
plt.show()

## Fit all 5 methods

K = 10 (true class count) for K-Means++ / GMM / Spectral Clustering.
Spectral Clustering uses a nearest-neighbors affinity (image-embedding
manifolds are typically not well captured by a single global RBF bandwidth).

In [ ]:
# The shared rule (tuning.select_all): K-Means k by silhouette (the number of classes when known),
# GMM by BIC, DBSCAN and HDBSCAN by the best silhouette with >= 2 clusters and <= 50% noise.
# Spectral clustering runs at the K-Means k.
selections = tuning.select_all(ds.data, k=ds.n_true_clusters, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Spectral"] = {"n_clusters": params["K-Means"]["n_clusters"], "affinity": "nearest_neighbors"}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

In [ ]:
results_df = gf.build_results_table(ds.name, ds.data, results, ds.target)
results_df.to_csv(results_dir("datasets") / "fashion_mnist_results.csv", index=False)
display(results_df[METRIC_COLS])
display(results_df[["method"] + EXTERNAL_COLS])

## Cluster visualizations vs. ground truth

In [ ]:
fig = gf.plot_cluster_grid(ds.data, results, y_true=ds.target, class_names=ds.target_names,
                           title="Fashion-MNIST (subsample): predicted clusters vs. ground truth (PCA 2D)")
plt.show()

## Qualitative check: example images per discovered cluster (K-Means++)

In [ ]:
km_labels = results["K-Means"].labels
fig, axes = plt.subplots(ds.n_true_clusters, 6, figsize=(10, 1.7 * ds.n_true_clusters))
for cluster_id in range(ds.n_true_clusters):
    idxs = np.where(km_labels == cluster_id)[0][:6]
    for col, ax in enumerate(axes[cluster_id]):
        if col < len(idxs):
            ax.imshow(ds.pixels[idxs[col]].reshape(28, 28), cmap="gray")
        ax.axis("off")
    axes[cluster_id, 0].set_ylabel(f"cluster {cluster_id}", rotation=0, labelpad=30, fontsize=8)
fig.suptitle("K-Means++ clusters: example member images")
plt.tight_layout()
plt.show()

## Discussion

- Even on a class-balanced subsample, Fashion-MNIST clustering is much harder
  than Wine/moons: visually similar classes (shirt/pullover/coat, sneaker/
  sandal/ankle-boot) overlap heavily in pixel/PCA space, so expect
  noticeably lower ARI/NMI across all methods than on the other datasets.
- **DBSCAN/HDBSCAN** often struggle here too (curse of dimensionality even
  at 50 PCA dims, plus uneven local density across visually-similar classes) —
  watch the `noise_fraction` and `n_clusters` columns to see whether they
  under- or over-segment relative to the true 10 classes.
- **Runtime** matters at this scale: compare `fit_seconds` across methods —
  Spectral Clustering's eigendecomposition and DBSCAN/HDBSCAN's neighbor
  search cost noticeably more than K-Means++/GMM even at just 3,000 points,
  which is exactly why the full 70k-point dataset was subsampled.
- Results here describe only the 3,000-point stratified subsample, not the
  full 70k-image dataset.